# Exploración del dataset local

Valida la estructura y calidad de `data/listings.csv` (dataset local recuperado, esquema Inside Airbnb más reciente). **No ejecuta limpieza ni escribe salidas**: solo inspecciona los datos para confirmar que se pueden reutilizar en el pipeline de limpieza.

In [1]:
import numpy as np
import pandas as pd

RUTA = "../data/listings.csv"

df = pd.read_csv(RUTA, low_memory=False)

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 92638
Columnas: 90


## 1. Columnas disponibles

In [2]:
for i, columna in enumerate(df.columns, start=1):
    print(f"{i}. {columna}")

1. id
2. listing_url
3. scrape_id
4. last_scraped
5. source
6. name
7. description
8. neighborhood_overview
9. picture_url
10. host_id
11. host_url
12. host_profile_id
13. host_profile_url
14. host_name
15. host_since
16. hosts_time_as_user_years
17. hosts_time_as_user_months
18. hosts_time_as_host_years
19. hosts_time_as_host_months
20. host_location
21. host_about
22. host_response_time
23. host_response_rate
24. host_acceptance_rate
25. host_is_superhost
26. host_thumbnail_url
27. host_picture_url
28. host_neighbourhood
29. host_listings_count
30. host_total_listings_count
31. host_verifications
32. host_has_profile_pic
33. host_identity_verified
34. neighbourhood
35. neighbourhood_cleansed
36. neighbourhood_group_cleansed
37. latitude
38. longitude
39. property_type
40. room_type
41. accommodates
42. bathrooms
43. bathrooms_text
44. bedrooms
45. beds
46. amenities
47. price
48. price_quote_checkin_date
49. price_quote_checkout_date
50. price_quote_total_price
51. price_quote_price_

## 2. Tipos de datos

In [3]:
print(df.dtypes)

id                                                int64
listing_url                                         str
scrape_id                                         int64
last_scraped                                        str
source                                              str
                                                 ...   
calculated_host_listings_count                    int64
calculated_host_listings_count_entire_homes       int64
calculated_host_listings_count_private_rooms      int64
calculated_host_listings_count_shared_rooms       int64
reviews_per_month                               float64
Length: 90, dtype: object


## 3. Valores faltantes

Resumen de columnas con datos faltantes (absoluto y porcentaje).

In [4]:
faltantes = pd.DataFrame({
    "Columna": df.columns,
    "Faltantes": df.isnull().sum().values,
    "Porcentaje": df.isnull().mean().values * 100
})

faltantes = (
    faltantes[faltantes["Faltantes"] > 0]
    .sort_values("Porcentaje", ascending=False)
    .reset_index(drop=True)
)

print("Columnas con faltantes:", len(faltantes))
display(faltantes.head(30))

Columnas con faltantes: 58


,Columna,Faltantes,Porcentaje
0,neighborhood_overview,92638,100.000000
1,host_since,92638,100.000000
2,host_neighbourhood,92638,100.000000
3,host_total_listings_count,92638,100.000000
4,neighbourhood,92638,100.000000
5,host_verifications,92638,100.000000
6,host_response_rate,92638,100.000000
7,host_acceptance_rate,92638,100.000000
8,host_thumbnail_url,92638,100.000000
9,host_response_time,92638,100.000000


## 4. Formato del precio

Verifica si `price` es texto con símbolo de moneda y prueba su conversión a numérico (sin modificar el dataset).

In [5]:
print("Tipo de price:", df["price"].dtype)
print(df["price"].head(10))

precio_num = pd.to_numeric(
    df["price"].astype(str).str.replace(r"[^\d.\-]", "", regex=True),
    errors="coerce"
)

print("\nConvertibles a numérico:", precio_num.notna().sum())
print("No convertibles:", precio_num.isna().sum())
print("Precios <= 0:", (precio_num <= 0).sum())

Tipo de price: str
0    $234.50
1    $127.00
2    $137.50
3    $606.67
4     $68.30
5     $49.06
6        NaN
7        NaN
8        NaN
9    $268.33
Name: price, dtype: str

Convertibles a numérico: 62240
No convertibles: 30398
Precios <= 0: 0


## 5. Columnas requeridas por el pipeline de limpieza

Confirma qué variables que usa el pipeline existen en este dataset local.

In [6]:
requeridas = [
    "id", "price", "latitude", "longitude", "accommodates",
    "bedrooms", "beds", "bathrooms", "bathrooms_text",
    "minimum_nights", "availability_365", "number_of_reviews",
    "reviews_per_month", "review_scores_rating", "amenities",
    "room_type", "property_type", "neighbourhood_cleansed",
    "host_is_superhost", "host_identity_verified", "instant_bookable",
    "host_response_rate", "host_acceptance_rate"
]

existentes = [c for c in requeridas if c in df.columns]
faltantes = [c for c in requeridas if c not in df.columns]

print("Existentes:", len(existentes), "de", len(requeridas))
print("\nFaltantes en el dataset local:", faltantes)

Existentes: 23 de 23

Faltantes en el dataset local: []


## 6. Geográfico

Comprueba si el dataset contiene varias ciudades.

In [7]:
for col in ["city", "market", "smart_location"]:
    if col in df.columns:
        print(f"--- {col} ---")
        print(df[col].value_counts(dropna=False).head(10))
        print()

## 7. IDs únicos y duplicados

In [8]:
print("Filas totales:", len(df))
print("IDs únicos:", df["id"].nunique())
print("IDs duplicados:", df["id"].duplicated().sum())
print("Filas duplicadas completas:", df.duplicated().sum())

Filas totales: 92638
IDs únicos: 92638
IDs duplicados: 0
Filas duplicadas completas: 0
